This directory is retained purely for archival purposes and is no longer updated. Please see the newly [consolidated LangChain documentation](https://docs.langchain.com/oss/python/langgraph/overview) for the most current information and resources.

# Corrective RAG (CRAG)

Corrective-RAG (CRAG) is a strategy for RAG that incorporates self-reflection / self-grading on retrieved documents.

In the paper [here](https://arxiv.org/pdf/2401.15884.pdf), a few steps are taken:

* If at least one document exceeds the threshold for relevance, then it proceeds to generation
* Before generation, it performs knowledge refinement
* This partitions the document into "knowledge strips"
* It grades each strip, and filters our irrelevant ones
* If all documents fall below the relevance threshold or if the grader is unsure, then the framework seeks an additional datasource
* It will use web search to supplement retrieval

We will implement some of these ideas from scratch using [LangGraph](https://langchain-ai.github.io/langgraph/):

* Let's skip the knowledge refinement phase as a first pass. This can be added back as a node, if desired.
* If *any* documents are irrelevant, let's opt to supplement retrieval with web search.
* We'll use [Tavily Search](https://python.langchain.com/v0.2/docs/integrations/tools/tavily_search/) for web search.
* Let's use query re-writing to optimize the query for web search.

![Screenshot 2024-04-01 at 9.28.30 AM.png](attachment:683fae34-980f-43f0-a9c2-9894bebd9157.png)

## Setup

First, let's download our required packages and set our API keys

In [1]:
%pip install -q langchain_community langchain-text-splitters tiktoken langchain-google-genai chromadb langchain langgraph tavily-python beautifulsoup4

In [2]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"] = "tvly-43yxvb-RucCoW0rnTmBp4Iq2LFMbyW43diX252ThevHcdTtAp"          # optional, only needed later in the tutorial
os.environ["USER_AGENT"] = "my-notebook"

In [3]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = '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'

In [4]:
%pip install -q langchain-huggingface sentence-transformers

<div class="admonition tip">
    <p class="admonition-title">Set up <a href="https://smith.langchain.com">LangSmith</a> for LangGraph development</p>
    <p style="padding-top: 5px;">
        Sign up for LangSmith to quickly spot issues and improve the performance of your LangGraph projects. LangSmith lets you use trace data to debug, test, and monitor your LLM apps built with LangGraph — read more about how to get started <a href="https://docs.smith.langchain.com">here</a>.
    </p>
</div>    

## Create Index

Let's index 3 blog posts.

In [5]:
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# tiktoken is only used here to count tokens for chunk sizes; no OpenAI key needed
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(docs_list)
print(f"Total chunks to index: {len(doc_splits)}")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Start from a clean collection so reruns don't create duplicates
try:
    Chroma(collection_name="rag-chroma", embedding_function=embeddings).delete_collection()
except Exception:
    pass

vectorstore = Chroma(
    collection_name="rag-chroma",
    embedding_function=embeddings,
)

batch_size = 90  # stay under the 100/min free-tier limit
for i in range(0, len(doc_splits), batch_size):
    batch = doc_splits[i : i + batch_size]
    for attempt in range(5):
        try:
            vectorstore.add_documents(batch)
            break
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print("Rate limited, waiting 60s...")
                time.sleep(60)
            else:
                raise
    print(f"Indexed {min(i + batch_size, len(doc_splits))}/{len(doc_splits)}")
    if i + batch_size < len(doc_splits):
        time.sleep(61)  # let the per-minute quota reset
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Done.")

/tmp/ipykernel_63707/2047549757.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


Total chunks to index: 187


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_63707/2047549757.py:27: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  Chroma(collection_name="rag-chroma", embedding_function=embeddings).delete_collection()


Indexed 90/187
Indexed 180/187
Indexed 187/187
Done.


In [6]:
import time
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI


class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

system = """You are a grader assessing relevance of a retrieved document to a user question.
If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant.
Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader


def grade_with_retry(inputs, retries=5):
    """Call the grader, waiting and retrying if the free tier rate-limits us."""
    for attempt in range(retries):
        try:
            return retrieval_grader.invoke(inputs)
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print("Rate limited, waiting 60s...")
                time.sleep(60)
            else:
                raise
    raise RuntimeError("Still rate limited after several retries.")


question = "agent memory"
docs = retriever.invoke(question)
print(f"Retrieved {len(docs)} documents")

doc_txt = docs[1].page_content if len(docs) > 1 else docs[0].page_content
print(grade_with_retry({"question": question, "document": doc_txt}))

Retrieved 2 documents
binary_score='yes'


In [7]:
### Generate

import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Prompt (same text as hub's "rlm/rag-prompt", written inline so no langchainhub or hub download is needed)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "human",
            """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:""",
        )
    ]
)

# LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)


# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run (retry if the free tier rate-limits us)
for attempt in range(5):
    try:
        generation = rag_chain.invoke(
            {"context": format_docs(docs), "question": question}
        )
        break
    except Exception as e:
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            print("Rate limited, waiting 60s...")
            time.sleep(60)
        else:
            raise

print(generation)

In an LLM-powered autonomous agent system, memory serves as a core component that allows agents to behave based on past experiences. Specifically, the memory stream acts as a long-term module, often utilizing an external database to record a comprehensive list of an agent's experiences in natural language. This mechanism enables agents to learn from past actions and interact effectively within their environments.


In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0)

# Prompt
system = """You are a question re-writer that converts an input question to a better version that is optimized
for web search. Look at the input and try to reason about the underlying semantic intent / meaning.
Return only the improved question, nothing else."""

re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        (
            "human",
            "Here is the initial question: \n\n {question} \n Formulate an improved question.",
        ),
    ]
)

question_rewriter = re_write_prompt | llm | StrOutputParser()

question = "agent memory"
print(question_rewriter.invoke({"question": question}))

How does memory work in autonomous AI agents and what are the different types of agent memory architectures?


In [9]:
### Search

from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(k=3)

/tmp/ipykernel_63707/730760015.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(k=3)


In [10]:
from typing import List

from typing_extensions import TypedDict


class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question: question
        generation: LLM generation
        web_search: whether to add search
        documents: list of documents
    """

    question: str
    generation: str
    web_search: str
    documents: List[str]

In [11]:
import time
from langchain_core.documents import Document


def invoke_with_retry(runnable, inputs, retries=5):
    """Call a chain, waiting and retrying if the Gemini free tier rate-limits us."""
    for attempt in range(retries):
        try:
            return runnable.invoke(inputs)
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print("Rate limited, waiting 60s...")
                time.sleep(60)
            else:
                raise
    raise RuntimeError("Still rate limited after several retries.")


def retrieve(state):
    """
    Retrieve documents

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, documents, that contains retrieved documents
    """
    print("---RETRIEVE---")
    question = state["question"]

    # Retrieval
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}


def generate(state):
    """
    Generate answer

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, generation, that contains LLM generation
    """
    print("---GENERATE---")
    question = state["question"]
    documents = state["documents"]

    # RAG generation
    generation = invoke_with_retry(
        rag_chain, {"context": format_docs(documents), "question": question}
    )
    return {"documents": documents, "question": question, "generation": generation}


def grade_documents(state):
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates documents key with only filtered relevant documents
    """

    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    question = state["question"]
    documents = state["documents"]

    # Score each doc
    filtered_docs = []
    web_search = "No"
    for d in documents:
        score = invoke_with_retry(
            retrieval_grader, {"question": question, "document": d.page_content}
        )
        grade = score.binary_score
        if grade.strip().lower() == "yes":
            print("---GRADE: DOCUMENT RELEVANT---")
            filtered_docs.append(d)
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---")
            web_search = "Yes"
            continue
    return {"documents": filtered_docs, "question": question, "web_search": web_search}


def transform_query(state):
    """
    Transform the query to produce a better question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates question key with a re-phrased question
    """

    print("---TRANSFORM QUERY---")
    question = state["question"]
    documents = state["documents"]

    # Re-write question
    better_question = invoke_with_retry(question_rewriter, {"question": question})
    return {"documents": documents, "question": better_question}


def web_search(state):
    """
    Web search based on the re-phrased question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates documents key with appended web results
    """

    print("---WEB SEARCH---")
    question = state["question"]
    documents = state["documents"]

    # Web search
    results = web_search_tool.invoke({"query": question})

    # Different Tavily tool versions return a list or a dict with a "results" key
    if isinstance(results, dict):
        results = results.get("results", [])
    web_results = "\n".join(d["content"] for d in results)
    documents.append(Document(page_content=web_results))

    return {"documents": documents, "question": question}


### Edges


def decide_to_generate(state):
    """
    Determines whether to generate an answer, or re-generate a question.

    Args:
        state (dict): The current graph state

    Returns:
        str: Binary decision for next node to call
    """

    print("---ASSESS GRADED DOCUMENTS---")
    web_search = state["web_search"]

    if web_search == "Yes":
        # Some documents were filtered out as not relevant,
        # so we rewrite the query and search the web
        print(
            "---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---"
        )
        return "transform_query"
    else:
        # We have relevant documents, so generate answer
        print("---DECISION: GENERATE---")
        return "generate"

In [12]:
from langgraph.graph import END, StateGraph, START

workflow = StateGraph(GraphState)

# Define the nodes
workflow.add_node("retrieve", retrieve)  # retrieve
workflow.add_node("grade_documents", grade_documents)  # grade documents
workflow.add_node("generate", generate)  # generate
workflow.add_node("transform_query", transform_query)  # transform_query
workflow.add_node("web_search_node", web_search)  # web search

# Build graph
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "transform_query": "transform_query",
        "generate": "generate",
    },
)
workflow.add_edge("transform_query", "web_search_node")
workflow.add_edge("web_search_node", "generate")
workflow.add_edge("generate", END)

# Compile
app = workflow.compile()

In [13]:
from pprint import pprint

# Run
inputs = {"question": "What are the types of agent memory?"}
for output in app.stream(inputs):
    for key, value in output.items():
        # Node
        pprint(f"Node '{key}':")
        # Optional: print full state at each node
        # pprint.pprint(value["keys"], indent=2, width=80, depth=None)
    pprint("\n---\n")

# Final generation
pprint(value["generation"])


---RETRIEVE---
"Node 'retrieve':"
'\n---\n'
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
"Node 'grade_documents':"
'\n---\n'
---GENERATE---
"Node 'generate':"
'\n---\n'
('The provided text identifies sensory memory as a type of memory, which '
 'includes subcategories such as iconic, echoic, and haptic memory. '
 'Additionally, the context mentions a "memory stream" as a long-term memory '
 'module used by generative agents to record experiences. No other specific '
 'types of agent memory are defined in the provided text.')


In [14]:
from pprint import pprint

# Run
inputs = {"question": "How does the AlphaCodium paper work?"}

final_state = None
for output in app.stream(inputs):
    for key, value in output.items():
        # Node
        pprint(f"Node '{key}':")
        # Optional: print the full state at each node
        # pprint(value, indent=2, width=80, depth=None)
        final_state = value
    pprint("\n---\n")

# Final generation
pprint(final_state["generation"])

---RETRIEVE---
"Node 'retrieve':"
'\n---\n'
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---
"Node 'grade_documents':"
'\n---\n'
---TRANSFORM QUERY---
"Node 'transform_query':"
'\n---\n'
---WEB SEARCH---
"Node 'web_search_node':"
'\n---\n'
---GENERATE---
"Node 'generate':"
'\n---\n'
('AlphaCodium is a code-oriented, iterative flow engineering approach that '
 'improves LLM performance by replacing standard prompting with a multi-stage, '
 'test-based process. The flow begins with a pre-processing phase to analyze '
 'the problem, generate test cases, and rank potential solutions. '
 'Subsequently, the code iterations phase repeatedly generates, executes, and '
 'fixes code against both public and AI-generated tests until a robust '
 'solution is achieved.')
